In [5]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import pandas as pd
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness, ContextPrecision, ContextRecall

from app.core.config import settings
from app.core.llm import get_llm
from app.rag.retrieval import retriever

In [7]:
golden_data = pd.read_csv("zipro_golden_dataset_25.csv")

print("Questions:", len(golden_data))
display(golden_data.head())

Questions: 25


,question,reference
0,Why do riders complain when a driver cancels a...,Riders report that accepted trips are cancelle...
1,How do app crashes affect customers while ente...,Customers report that the app closes during pi...
2,Why do customers complain about refunds after ...,Customers report that refunds remain pending f...
3,What information do customers want while waiti...,"Customers request clear refund status, process..."
4,What happens when a driver is marked as arrive...,Some riders describe a driver shown as nearby ...


In [8]:
rows = []

for question, reference in zip(golden_data["question"], golden_data["reference"]):
    documents = retriever.search(question)
    contexts = [doc.page_content for doc in documents[:5]]

    if not contexts:
        answer = "No customer evidence found."
    else:
        prompt = f"""Answer the question using only the feedback below.
Do not invent numbers, causes or evidence. Say when evidence is insufficient.

Feedback:
{chr(10).join(contexts)}

Question: {question}"""
        answer = get_llm().invoke(prompt).content

    rows.append({
        "question": question,
        "reference": reference,
        "answer": answer,
        "contexts": contexts
    })
    print("Answered:", question)

print("Questions processed:", len(rows))

Answered: Why do riders complain when a driver cancels after accepting a trip?
Answered: How do app crashes affect customers while entering pickup details?
Answered: Why do customers complain about refunds after cancelled trips?
Answered: What information do customers want while waiting for a refund?
Answered: What happens when a driver is marked as arrived but the rider cannot find the car?
Answered: Why do customers complain that their pickup pin is incorrect?
Answered: What do customers report about duplicate card charges?
Answered: Why do customers say driver GPS tracking is unreliable?
Answered: What issues occur when ride OTP verification is delayed?
Answered: What challenges do customers face when contacting support?


Cohere reranking failed; using retrieved documents
Traceback (most recent call last):
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\retrieval.py", line 86, in search
    return rerank_feedback(question, documents)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\reranker.py", line 26, in rerank_feedback
    return list(get_reranker().compress_documents(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-packages\langchain_cohere\rerank.py", line 158, in compress_documents
    for res in self.rerank(documents, query):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-pac

Answered: How do driver cancellations and app crashes disrupt customers differently?
Answered: Compare the customer frustrations caused by driver cancellations and refund delays.


Cohere reranking failed; using retrieved documents
Traceback (most recent call last):
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\retrieval.py", line 86, in search
    return rerank_feedback(question, documents)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\reranker.py", line 26, in rerank_feedback
    return list(get_reranker().compress_documents(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-packages\langchain_cohere\rerank.py", line 158, in compress_documents
    for res in self.rerank(documents, query):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-pac

Answered: How are app crashes and refund delays different customer experience problems?
Answered: How is a driver no-show different from a driver cancellation?
Answered: How do pickup-pin errors and GPS tracking problems affect different stages of a ride?
Answered: What is the difference between a duplicate payment complaint and a refund delay complaint?


Cohere reranking failed; using retrieved documents
Traceback (most recent call last):
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\retrieval.py", line 86, in search
    return rerank_feedback(question, documents)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\app\rag\reranker.py", line 26, in rerank_feedback
    return list(get_reranker().compress_documents(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-packages\langchain_cohere\rerank.py", line 158, in compress_documents
    for res in self.rerank(documents, query):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\Videos\machine learning projects 2026 resume\Zipro_Voice_of_Customer_Copilot\backend\venv\Lib\site-pac

Answered: Why might scheduled-ride cancellations be especially disruptive?
Answered: What customer evidence suggests the promotion experience needs clearer rules?
Answered: How do fare-estimate complaints differ from missing receipt complaints?
Answered: How do poor driver conduct and slow support responses combine to damage confidence?
Answered: What is a practical first improvement to the driver cancellation recovery experience?
Answered: If Zipro investigates payment reliability, how should it separate suspected duplicate charges from failed authorizations?
Answered: How could Zipro prioritize airport pickup confusion versus general pickup-pin errors without inventing complaint counts?
Answered: What should Zipro verify before claiming that a frozen driver map is caused by a GPS outage?
Answered: How should Zipro design a first test of clearer scheduled-ride reliability messaging?
Questions processed: 25


In [9]:
from openai import AsyncOpenAI

judge = llm_factory(
    "gpt-4.1-mini",
    client=AsyncOpenAI(api_key=settings.OPENAI_API_KEY)
)

faithfulness = Faithfulness(llm=judge)
precision = ContextPrecision(llm=judge)
recall = ContextRecall(llm=judge)

In [11]:
results = []

for row in rows:

    question = row["question"]
    answer = row["answer"]
    contexts = row["contexts"]
    reference = row["reference"]

    if not contexts:
        continue

    # Evaluate faithfulness
    f_score = await faithfulness.ascore(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts
    )

    # Evaluate context precision
    p_score = await precision.ascore(
        user_input=question,
        reference=reference,
        retrieved_contexts=contexts
    )

    # Evaluate context recall
    r_score = await recall.ascore(
        user_input=question,
        reference=reference,
        retrieved_contexts=contexts
    )

    results.append({
        "question": question,
        "faithfulness": f_score.value,
        "context_precision": p_score.value,
        "context_recall": r_score.value
    })

    print("Evaluated:", question)

evaluation_results = pd.DataFrame(results)

evaluation_results

Evaluated: Why do riders complain when a driver cancels after accepting a trip?
Evaluated: How do app crashes affect customers while entering pickup details?
Evaluated: Why do customers complain about refunds after cancelled trips?
Evaluated: What information do customers want while waiting for a refund?
Evaluated: What happens when a driver is marked as arrived but the rider cannot find the car?
Evaluated: Why do customers complain that their pickup pin is incorrect?
Evaluated: What do customers report about duplicate card charges?
Evaluated: Why do customers say driver GPS tracking is unreliable?
Evaluated: What issues occur when ride OTP verification is delayed?
Evaluated: What challenges do customers face when contacting support?
Evaluated: How do driver cancellations and app crashes disrupt customers differently?
Evaluated: Compare the customer frustrations caused by driver cancellations and refund delays.
Evaluated: How are app crashes and refund delays different customer experie

,question,faithfulness,context_precision,context_recall
0,Why do riders complain when a driver cancels a...,1.000000,1.000000,1.0
1,How do app crashes affect customers while ente...,1.000000,1.000000,1.0
2,Why do customers complain about refunds after ...,1.000000,1.000000,1.0
3,What information do customers want while waiti...,1.000000,1.000000,1.0
4,What happens when a driver is marked as arrive...,1.000000,1.000000,0.5
5,Why do customers complain that their pickup pi...,0.750000,1.000000,1.0
6,What do customers report about duplicate card ...,1.000000,1.000000,0.0
7,Why do customers say driver GPS tracking is un...,1.000000,1.000000,1.0
8,What issues occur when ride OTP verification i...,1.000000,1.000000,1.0
9,What challenges do customers face when contact...,1.000000,1.000000,1.0


In [13]:
metrics = ["faithfulness", "context_precision", "context_recall"]

print("Average RAGAS scores:")
print(evaluation_results[metrics].mean().round(3))

Average RAGAS scores:
faithfulness         0.867
context_precision    0.777
context_recall       0.720
dtype: float64
